# PIKAN prediction for the semi-infinite-domain problem

In [1]:
import numpy as np

In [2]:
from pathlib import Path
import sys
from importlib import reload

import matplotlib.pyplot as plt
import numpy as np
import torch

notebook_dir = Path.cwd().resolve()
repo_root = next(
    (path for path in [notebook_dir, *notebook_dir.parents] if (path / "utils").is_dir()),
    notebook_dir,
)
utilities_dir = repo_root / "utils"
if str(utilities_dir) not in sys.path:
    sys.path.insert(0, str(utilities_dir))

import pinns_infinite
import pinns_semi_infinite
import semi_infinite
reload(semi_infinite)
reload(pinns_infinite)
reload(pinns_semi_infinite)

from pinns_semi_infinite import build_models_KAN, set_seed, train_dual_network_semi_inf
from semi_infinite import (
    analytical_solution_semi_inf,
    coefficient_semi_inf,
    evaluate_model_semi_inf,
)

set_seed(42)
torch.set_default_dtype(torch.float32)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

/home/orincon/miniconda3/envs/PIKAN-unbounded-domains-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


## Tuned PIKAN configuration

In [3]:
import pandas as pd

# Pick the most recent semi-infinite KAN optimization run that has saved weights.
tunning_dir = repo_root / "main" / "02_hyperparameter_tunning"
semi_infinite_studies = sorted(
    d for d in tunning_dir.glob("results_kan_semi_infinite_optuna_*")
    if (d / "summary_metrics.csv").exists()
)
if not semi_infinite_studies:
    raise FileNotFoundError(
        "No semi-infinite optimization study with a saved summary_metrics.csv was found. "
        "Rerun 06_kan_semi_infinite_domain_optimization.ipynb (weight-saving is now enabled) "
        "before running this notebook."
    )
optimization_dir = semi_infinite_studies[-1]

summary = pd.read_csv(optimization_dir / "summary_metrics.csv")
best_trial = summary.loc[summary["mean_global_error"].idxmin()]
best_trial_dir = optimization_dir / "KAN" / str(best_trial["timestamp"])

config = {
    "mode": "load",
    "hidden_layers": int(best_trial["hidden_layers"]),
    "hidden_units": int(best_trial["hidden_units"]),
    "grid_size": int(best_trial["grid_size"]),
    "spline_order": int(best_trial["spline_order"]),
    "adam_lr": float(best_trial["adam_lr"]),
    "sampling": str(best_trial["sampling"]),
    "sigma": float(best_trial["sigma"]),
    "exp_scale": float(best_trial["exp_scale"]),
    "n_obs_u": int(best_trial["n_obs_u"]),
    "n_boundary_u": int(best_trial["n_obs_u"]),
    "n_obs_k": int(best_trial["n_obs_k"]),
    "n_pde": int(best_trial["n_pde"]),
    "seed": int(best_trial["seed"]),
    "pde_alpha": float(best_trial["pde_alpha"]),
    "pde_beta": float(best_trial["pde_beta"]),
    "epsilon": float(best_trial["epsilon"]),
    "train_domain": (-5.0, 5.0, -5.0, 0.0),
    "eval_domain": (-10.0, 10.0, -10.0, 0.0),
}

print("Selected best semi-infinite KAN optimization trial:")
print(f"Study directory: {optimization_dir}")
print(f"Trial directory: {best_trial_dir}")
print(f"Optimization mean global error: {best_trial['mean_global_error']:.12e}")
for name, value in config.items():
    print(f"{name}: {value}")

Selected best semi-infinite KAN optimization trial:
Study directory: /home/orincon/unbounded-domains/main/02_hyperparameter_tunning/results_kan_semi_infinite_optuna_2026-09-20_20-46-49
Trial directory: /home/orincon/unbounded-domains/main/02_hyperparameter_tunning/results_kan_semi_infinite_optuna_2026-09-20_20-46-49/KAN/2026-09-21_01-10-59
Optimization mean global error: 8.376314407819e-04
mode: load
hidden_layers: 3
hidden_units: 15
grid_size: 5
spline_order: 3
adam_lr: 0.01
sampling: gaussian_exponential
sigma: 5.5
exp_scale: 7.0
n_obs_u: 100
n_boundary_u: 100
n_obs_k: 100
n_pde: 1000
seed: 42
pde_alpha: 0.5
pde_beta: 5.0
epsilon: 1.0
train_domain: (-5.0, 5.0, -5.0, 0.0)
eval_domain: (-10.0, 10.0, -10.0, 0.0)


## Build the KAN models

In [4]:
model_u, model_k = build_models_KAN(
    device=device,
    hidden_layers=config["hidden_layers"],
    hidden_units=config["hidden_units"],
    grid_size=config["grid_size"],
    spline_order=config["spline_order"],
)

print(model_u)
print(model_k)

KAN(
  (layers): ModuleList(
    (0-3): 4 x KANLinear(
      (base_activation): SiLU()
    )
  )
)
KAN(
  (layers): ModuleList(
    (0-3): 4 x KANLinear(
      (base_activation): SiLU()
    )
  )
)


## Load the Gaussian semi-infinite PIKAN

In [5]:
import shutil

results_dir = repo_root / "main" / "03_individual_prediction" / "results"
results_dir.mkdir(parents=True, exist_ok=True)

model_u_path = best_trial_dir / "model_u.pt"
model_k_path = best_trial_dir / "model_k.pt"

if not model_u_path.exists() or not model_k_path.exists():
    raise FileNotFoundError(
        f"Optimization weights were not found in {best_trial_dir}"
    )

model_u.load_state_dict(torch.load(model_u_path, map_location=device))
model_k.load_state_dict(torch.load(model_k_path, map_location=device))
model_u.eval()
model_k.eval()

local_model_u_path = results_dir / "pikan_semi_infinite_exponential_model_u.pt"
local_model_k_path = results_dir / "pikan_semi_infinite_exponential_model_k.pt"
shutil.copy2(model_u_path, local_model_u_path)
shutil.copy2(model_k_path, local_model_k_path)

print(f"Loaded optimization weights from: {best_trial_dir}")
print(f"Stored u weights at: {local_model_u_path}")
print(f"Stored k weights at: {local_model_k_path}")
print(model_u)
print(model_k)

Loaded optimization weights from: /home/orincon/unbounded-domains/main/02_hyperparameter_tunning/results_kan_semi_infinite_optuna_2026-09-20_20-46-49/KAN/2026-09-21_01-10-59
Stored u weights at: /home/orincon/unbounded-domains/main/03_individual_prediction/results/pikan_semi_infinite_exponential_model_u.pt
Stored k weights at: /home/orincon/unbounded-domains/main/03_individual_prediction/results/pikan_semi_infinite_exponential_model_k.pt
KAN(
  (layers): ModuleList(
    (0-3): 4 x KANLinear(
      (base_activation): SiLU()
    )
  )
)
KAN(
  (layers): ModuleList(
    (0-3): 4 x KANLinear(
      (base_activation): SiLU()
    )
  )
)


## Evaluate against the analytical solution

In [6]:
evaluation = evaluate_model_semi_inf(
    model_u=model_u,
    model_k=model_k,
    analytical_solution=analytical_solution_semi_inf,
    coefficient=coefficient_semi_inf,
    train_domain=config["train_domain"],
    eval_domain=config["eval_domain"],
    n_grid=400,
    alpha=config["pde_alpha"],
    beta=config["pde_beta"],
    epsilon=config["epsilon"],
    device=device,
    verbose=True,
)

metric_names = [
    "err_u_global",
    "err_k_global",
    "err_u_inside",
    "err_k_inside",
    "err_u_outside",
    "err_k_outside",
]

metrics = {
    name: float(evaluation[name])
    for name in metric_names
}

# Mean global error
metrics["err_mean_global"] = (
    metrics["err_u_global"] +
    metrics["err_k_global"]
) / 2.0

local_checkpoint = results_dir / "pikan_semi_infinite_exponential_weights.pt"

torch.save(
    {
        "model_u": model_u.state_dict(),
        "model_k": model_k.state_dict(),
        "config": config,
        "metrics": metrics,
        "source_trial": str(best_trial_dir),
        "optimization_mean_global_error": float(best_trial["mean_global_error"]),
    },
    local_checkpoint,
)

absolute_difference = abs(
    metrics["err_mean_global"] - float(best_trial["mean_global_error"])
)

print(f"Stored combined checkpoint at: {local_checkpoint}")
print(f"Optimization mean global error: {best_trial['mean_global_error']:.12e}")
print(f"Notebook mean global error: {metrics['err_mean_global']:.12e}")
print(f"Absolute difference: {absolute_difference:.12e}")

metrics

Semi-infinite-domain spatial generalization (MAE)
u (global): 1.363e-03
k (global): 3.125e-04
u (inside): 1.056e-03
k (inside): 2.274e-04
u (outside): 1.465e-03
k (outside): 3.408e-04
Stored combined checkpoint at: /home/orincon/unbounded-domains/main/03_individual_prediction/results/pikan_semi_infinite_exponential_weights.pt
Optimization mean global error: 8.376314407819e-04
Notebook mean global error: 8.376314407820e-04
Absolute difference: 6.917209860458e-17


{'err_u_global': 0.0013627921535614583,
 'err_k_global': 0.00031247072800248015,
 'err_u_inside': 0.0010561588147017821,
 'err_k_inside': 0.00022739794864556258,
 'err_u_outside': 0.0014650032665146837,
 'err_k_outside': 0.0003408283211214527,
 'err_mean_global': 0.0008376314407819692}